# Transition volcano with an effect size on x

The same cells and state sets as `08` figure 2 (the `TOP_N` best bstm sets per task and phenotype),
drawn like a differential-expression volcano.

**Why `08`'s volcano is a narrow V.** Its x axis is a correlation r, which already divides out each
cell's noise. With n fixed, p depends on |r| alone, so every dot lands on one curve. A gene volcano
puts a *raw* effect on x (log2 fold change) and takes p from a test that uses each gene's own variance.
Two genes with the same fold change can then have very different p, and the points spread into a cloud.

`X_AXIS` picks the x axis. All three adjust for `COVARIATES` by regression:

| `X_AXIS` | x | p from |
|---|---|---|
| `"log2fc"` | log2( mean cell in the elevated group / mean in Normal ), elevated = `CASE_FROM` and above | t-test of the group term |
| `"slope"` | change in the cell's probability per HADS level (raw units) | t-test of the slope |
| `"r"` | partial Spearman r, exactly as in `08` | the same |

A point is **up** or **down** when it survives BH-FDR at `ALPHA` over the panel *and* clears `FC_LINE`.
Cells present in fewer than `MIN_PRESENT` of subjects are dropped first: their means sit near zero, so a
ratio of two of them is mostly noise. This is the same reason gene pipelines filter low counts.

In [ ]:
import os
from pathlib import Path

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
CAMCAN = Path(os.environ.get("CAMCAN_PHENO",
                             "/project/6008063/tamires/cohorts/camcan/dataman/useraccess/"
                             "opendata/paule_toussaint_camcan01870"))
PHENO_FILES = [(CAMCAN / "approved_data.tsv", "\t"),   # HADS
               (CAMCAN / "standard_data.csv", ",")]     # Age, Sex
PHENO_ID = "CCID"

# label -> column. Labels are what the figures say.
TARGETS = {"anxiety":    "additional_HADS_anx_category",
           "depression": "additional_HADS_dep_category"}
ORDINAL_LEVELS = ["Normal", "Mild", "Moderate", "Severe"]   # low -> high

# task label in the *_selection trees -> task value inside the transition tables
TASKS = {"rest": "rest", "movie": "movie"}                  # matched case-insensitively
COHORTS = ["camcan", "camcan_rest"]                         # cohort= dirs to read

MODEL = "ridge"                                  # ridge | hgb | lgbm | logistic
ARM = {"bstm": "cells [joint]", "fcm": "edges"}  # the hypothesis arm per tree

TOP_N = 10                                       # state sets per volcano panel
COVARIATES = ["Age", "Sex", "n_transitions"]
CATEGORICAL = ["Sex"]
ALPHA = 0.05                                     # BH-FDR, per panel
SAVE_DIR = None                                  # e.g. Path("figures/08") to save PNG + SVG

X_AXIS = "log2fc"                                # "log2fc" | "slope" | "r"
CASE_FROM = "Mild"                               # log2fc: this level and above = elevated
FC_LINE = {"log2fc": 0.25, "slope": None, "r": None}[X_AXIS]   # |x| cut, in x units; None = FDR only
MIN_PRESENT = 0.10                               # drop cells non-zero in < this share of subjects
LABEL_TOP = 6                                    # cells annotated per panel

In [ ]:
import sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats

INK, INK2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#a3a29d", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "axes.titlesize": 11,
    "axes.titlecolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "font.size": 9, "legend.frameon": False})

def hive(path, key):
    """One `key=value` partition from a path, by name."""
    return next((p.split("=", 1)[1] for p in Path(path).parts if p.startswith(f"{key}=")), None)

def fmt_window(w):
    return str(int(w)) if float(w).is_integer() else str(w)

def set_label(r):
    """A readable name for one summary row: state set or static-FC atlas."""
    if r.get("tree") == "fcm":
        return r["atlas"]
    w = "1 TR" if float(r["window_s"]) < 0 else f"{fmt_window(r['window_s'])}s"
    return f"{r['states']} · {r['atlas']} · {w}"

def save(fig, name):
    if SAVE_DIR:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        for ext in ("png", "svg"):
            fig.savefig(Path(SAVE_DIR) / f"{name}.{ext}", dpi=200, bbox_inches="tight")

def design(cov, intercept=True):
    """Covariates -> numeric matrix: categoricals one-hot, gaps mean-filled."""
    X = pd.get_dummies(cov, columns=[c for c in CATEGORICAL if c in cov],
                       drop_first=True).astype(float)
    X = X.fillna(X.mean()).to_numpy()
    return np.column_stack([np.ones(len(X)), X]) if intercept else X
UP, DOWN, NS = "#e34948", "#2a78d6", "#c3c2bd"   # the diverging pair, grey for nothing

In [ ]:
def read_summaries(tree):
    files = sorted((ROOT / f"{tree}_selection").glob("task=*/target=*/summary.csv"))
    if not files:
        raise SystemExit(f"no summary.csv under {ROOT / f'{tree}_selection'}")
    return pd.concat([pd.read_csv(p).assign(tree=tree, task=hive(p, "task"),
                                            target=hive(p, "target"))
                      for p in files], ignore_index=True)

def read_pheno(covariates):
    frames = [pd.read_csv(p, sep=s, dtype={PHENO_ID: str}) for p, s in PHENO_FILES]
    ph = frames[0]
    for f in frames[1:]:
        ph = ph.merge(f.drop(columns=[c for c in f if c in ph and c != PHENO_ID]),
                      on=PHENO_ID, how="outer")
    lower = {c.lower(): c for c in ph.columns}              # Age vs age
    ph = ph.rename(columns={lower[c.lower()]: c for c in covariates if c.lower() in lower})
    ph["sub"] = ph[PHENO_ID].astype(str).str.strip().str.upper()
    code = {lvl: k for k, lvl in enumerate(ORDINAL_LEVELS)}
    for lab, col in TARGETS.items():
        ph[lab] = ph[col].astype("string").str.strip().map(code).astype(float)
    for c in set(covariates) - set(CATEGORICAL) - {"n_transitions"}:
        ph[c] = pd.to_numeric(ph[c], errors="coerce")
    return ph[["sub", *TARGETS, *(c for c in covariates if c in ph)]]

def load_set(atlas, window_s, states, task):
    """One state set's transition table for one task, one row per subject."""
    base = (ROOT / "transitions" / f"atlas={atlas}" / f"window_s={fmt_window(window_s)}"
            / f"states={states}")
    files = [p for p in base.rglob("subjects.parquet")
             if hive(p, "cohort") in COHORTS
             and (hive(p, "task") or task).lower() == TASKS[task].lower()]
    if not files:
        raise SystemExit(f"no transition table for {base} task={task}")
    d = pd.concat(map(pd.read_parquet, files), ignore_index=True)
    d = d[d["task"].astype(str).str.lower() == TASKS[task].lower()]
    d["sub"] = d["sub"].astype(str).str.strip().str.upper()
    if d["sub"].duplicated().any():
        raise SystemExit(f"{states} {task}: a subject appears twice -- narrow COHORTS")
    return d

def cell_cols(d):
    return [c for c in d.columns if "->" in c]

def as_p_cond(d, K):
    """p_joint -> Markov p(j | i): back to counts, then renormalise each row."""
    cols = cell_cols(d)
    M = d[cols].to_numpy(float).reshape(len(d), K, K) * d["n_transitions"].to_numpy(float)[:, None, None]
    s = M.sum(2, keepdims=True)
    return pd.DataFrame(np.divide(M, s, out=np.zeros_like(M), where=s > 0).reshape(len(d), -1),
                        columns=cols, index=d.index)

def partial_spearman(A, y, cov):
    """Rank everything, residualise on the covariates, correlate. Returns r, p."""
    X = design(cov)
    R = np.column_stack([stats.rankdata(y), np.apply_along_axis(stats.rankdata, 0, A)])
    R = R - X @ np.linalg.lstsq(X, R, rcond=None)[0]
    R = (R - R.mean(0)) / np.where(R.std(0) > 0, R.std(0), np.nan)
    r = R[:, 1:].T @ R[:, 0] / len(R)
    df = len(R) - X.shape[1] - 1
    p = 2 * stats.t.sf(np.abs(r * np.sqrt(df / (1 - r ** 2))), df)
    return r, p

def top_sets(task, lab, n=TOP_N):
    """The n best bstm state sets of one ranking, as summary.csv rows."""
    g = summ[(summ["tree"] == "bstm") & (summ["task"] == task) & (summ["phenotype"] == lab)
             & (summ["arm"] == ARM["bstm"])]
    return (g.sort_values("mean", ascending=False)
             .drop_duplicates(["atlas", "window_s", "states"]).head(n).reset_index(drop=True))

def merged(s, task, lab):
    """One state set's table joined to phenotype, cells as ARM asks for."""
    t = load_set(s["atlas"], s["window_s"], s["states"], task)
    if "[cond]" in ARM["bstm"]:
        t[cell_cols(t)] = as_p_cond(t, int(s["K"]))
    d = t.merge(pheno, on="sub", suffixes=("", "_pheno")).dropna(
        subset=[lab, *(c for c in COVARIATES if c not in CATEGORICAL)])
    return d, cell_cols(t)

label_of = {col: lab for lab, col in TARGETS.items()}
summ = pd.concat([read_summaries("bstm"), read_summaries("fcm")], ignore_index=True)
summ = summ[(summ["model"] == MODEL) & summ["target"].isin(label_of)
            & summ["task"].isin(TASKS)].assign(phenotype=lambda d: d["target"].map(label_of))
summ["label"] = summ.apply(set_label, axis=1)

pheno = read_pheno(COVARIATES)
print(f"phenotype: {len(pheno):,} subjects")

In [ ]:
def effects(A, y, cov):
    """Effect of the phenotype on every column of A, adjusted for cov. Returns (x, p)."""
    if X_AXIS == "r":
        return partial_spearman(A, y, cov)
    case = ORDINAL_LEVELS.index(CASE_FROM)
    g = (y >= case).astype(float) if X_AXIS == "log2fc" else y
    X = np.column_stack([g, design(cov)])
    B = np.linalg.lstsq(X, A, rcond=None)[0]
    df = len(A) - X.shape[1]
    se = np.sqrt(((A - X @ B) ** 2).sum(0) / df * np.linalg.pinv(X.T @ X)[0, 0])
    p = 2 * stats.t.sf(np.abs(B[0] / se), df)
    if X_AXIS == "slope":
        return B[0], p
    mu0 = A[g == 0].mean(0)                       # Normal-group mean
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.log2(np.clip(mu0 + B[0], 1e-12, None) / mu0), p

def scan(task, lab):
    out = []
    for rank, s in top_sets(task, lab).iterrows():
        d, cols = merged(s, task, lab)
        A = d[cols]
        A = A.loc[:, (A > 0).mean() >= MIN_PRESENT]
        x, p = effects(A.to_numpy(float), d[lab].to_numpy(float), d[COVARIATES])
        out.append(pd.DataFrame({"rank": rank + 1, "set": s["label"], "transition": A.columns,
                                 "x": x, "p": p, "n": len(d)}))
    v = pd.concat(out, ignore_index=True).dropna(subset=["x", "p"])
    v["p_fdr"] = stats.false_discovery_control(v["p"], method="bh")
    big = v["x"].abs() >= (FC_LINE or 0)
    v["call"] = np.select([(v["p_fdr"] < ALPHA) & big & (v["x"] > 0),
                           (v["p_fdr"] < ALPHA) & big & (v["x"] < 0)], ["up", "down"], "not sig")
    return v.assign(task=task, phenotype=lab)

volc = pd.concat([scan(task, lab) for task in TASKS for lab in TARGETS], ignore_index=True)
volc.nsmallest(10, "p")[["task", "phenotype", "rank", "set", "transition", "x", "p", "p_fdr", "call"]]

In [ ]:
XLABEL = {"log2fc": f"log2 fold change  ({CASE_FROM}+ vs Normal, adjusted)",
          "slope": "Δ transition probability per HADS level  (adjusted)",
          "r": "partial Spearman r"}[X_AXIS]

def plot_volcano(ax, v):
    y = -np.log10(v["p"])
    for call, c in (("not sig", NS), ("down", DOWN), ("up", UP)):
        m = v["call"] == call
        ax.scatter(v["x"][m], y[m], s=18, color=c, alpha=.7 if call != "not sig" else .5, lw=0,
                   label=f"{call.capitalize()} ({m.sum():,})")
    for t in v.nsmallest(LABEL_TOP, "p").itertuples():
        ax.annotate(f"{t.transition} (#{t.rank})", (t.x, -np.log10(t.p)), xytext=(4, 3),
                    textcoords="offset points", fontsize=7, color=INK2)
    ax.axhline(-np.log10(.05), color=MUTED, ls=":", lw=.8)
    if (v["p_fdr"] < ALPHA).any():
        ax.axhline(-np.log10(v.loc[v["p_fdr"] < ALPHA, "p"].max()), color=INK2, ls="--", lw=.8)
    for s in (-1, 1):
        if FC_LINE:
            ax.axvline(s * FC_LINE, color=MUTED, ls=":", lw=.8)
    ax.axvline(0, color=MUTED, lw=.6)
    lim = 1.08 * v["x"].abs().max()
    ax.set_xlim(-lim, lim)
    ax.set_title(f"{v['task'].iloc[0]} · {v['phenotype'].iloc[0]}   (n≈{int(v['n'].median())})", fontsize=10)
    calls = ax.legend(loc="upper center", ncol=3, fontsize=8,
                      handletextpad=.2, markerscale=1.4)
    ax.add_artist(calls)
    key = v.drop_duplicates("rank")
    ax.legend([Line2D([], [], ls="none")] * len(key),
              [f"#{r}  {s}" for r, s in zip(key["rank"], key["set"])],
              loc="upper center", bbox_to_anchor=(.5, -.16), ncol=2, fontsize=7,
              handlelength=0, handletextpad=0, columnspacing=1.5)

fig, axes = plt.subplots(len(TASKS), len(TARGETS), squeeze=False,
                         figsize=(6.5 * len(TARGETS), 6.3 * len(TASKS)))
for (i, task), (j, lab) in ((a, b) for a in enumerate(TASKS) for b in enumerate(TARGETS)):
    plot_volcano(axes[i, j], volc[(volc["task"] == task) & (volc["phenotype"] == lab)])
for ax in axes[:, 0]:
    ax.set_ylabel(r"$-\log_{10}$ p")
for ax in axes[-1]:
    ax.set_xlabel(XLABEL)
fig.tight_layout(h_pad=3)
save(fig, f"fig2_volcano_{X_AXIS}")

### Reading it

* **The dashed line** is the BH-FDR cut over the panel. **Dotted lines** are p = 0.05 uncorrected
  and ±`FC_LINE`.
* **Effect sizes:** `log2fc` is a ratio, so a cell that moves from 0.010 to 0.012 reads like one
  that moves from 0.10 to 0.12. That is why it compares fairly across K. `slope` keeps raw units,
  so cells from small-K sets (larger probabilities) spread wider.
* **Labels:** `#k` is the set's rank in that panel's `bstm_selection` ranking, keyed under each panel.
* **The same cells, a different x:** the up and down calls apply the same FDR as `08` plus an
  effect-size cut, so they can only be fewer than `08`'s FDR hits for `r`. For `log2fc` they can
  differ, because the test is on the group contrast, not the ordinal.